# Lab 1.3 &mdash; Tool Descriptions Are Instructions

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Run the same agent twice: same model, same functions, different descriptions
- Measure the difference on a small eval set instead of asserting it
- Make the agent return a typed object rather than a paragraph

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 1 labs work one case: a small tech-support ticket queue.
> What you build in each lab is picked up by the next one.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# A small tech-support ticket queue. Ordinary rules on purpose: the only new thing in these
# five labs is LangChain. Nothing here is real data and nothing leaves this notebook.

TICKETS = {
    "TCK-4001": {"customer": "Priya Nair",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "VPN-513",
                 "text": "Cannot connect since the upgrade. Error VPN-513."},
    "TCK-4002": {"customer": "Rahul Menon",  "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Monthly export finishes but the PDF is blank."},
    "TCK-4003": {"customer": "Anita Sharma", "product": "Reports",    "version": "3.9.1",
                 "severity": "medium", "error_code": None,
                 "text": "It is just slow today. Nothing else to add."},
    "TCK-4004": {"customer": "Vikram Rao",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "SEC-900",
                 "text": "Got a login alert from a country I have never visited."},
    "TCK-4005": {"customer": "Priya Nair",   "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Same blank PDF as my colleague reported."},
}

# The runbook: what support is allowed to do about each error code.
RUNBOOK = {
    "VPN-513": "Certificate pinning changed in 4.2. Have the user clear the local trust store "
               "and re-enrol. Five minutes, no data loss. Support may do this without approval.",
    "APP-002": "Known defect in 3.9.1, fixed in 3.9.2. Advise the upgrade. Do not issue a refund "
               "for this and do not raise a new defect -- link the existing one.",
    "SEC-900": "Possible credential compromise. Escalate to the security desk immediately. "
               "Support must not resolve, close or advise the customer directly.",
}

# Which error codes may an agent resolve on its own, and which need a human?
MUST_ESCALATE = {"SEC-900"}

print(f"{len(TICKETS)} tickets, {len(RUNBOOK)} runbook entries loaded")

## Concept

The model never sees your function. It sees a **name, a description and a parameter schema** &mdash;
and it picks from those alone.

So a tool description is not documentation. It is the prompt that decides whether the right tool
gets called, and it is the cheapest accuracy in this course: no model change, no extra call, no
new framework.

This lab measures it. Same model, same three functions, two sets of words.

## Section 1 &mdash; The same tools, described two ways

The functions and the scoring harness are given. Your job is the words.

Write descriptions that answer the two questions a model actually has: **what does this return**,
and **when should I reach for it rather than the other one?**

In [ ]:
from langchain_core.tools import StructuredTool

def _ticket(ref: str) -> str:
    t = TICKETS.get(ref)
    return json.dumps({"ref": ref, **t}) if t else f"no ticket {ref!r}"

def _runbook(error_code: str) -> str:
    return RUNBOOK.get(error_code, f"no runbook entry for {error_code!r}")

def _similar(error_code: str) -> str:
    return json.dumps([r for r, t in TICKETS.items() if t["error_code"] == error_code])

OPS = {"lookup_ticket": _ticket, "runbook_for": _runbook, "find_similar": _similar}

def build_tools(descriptions: dict) -> list:
    """Three real LangChain tools over the same three functions, with the words you choose."""
    return [StructuredTool.from_function(func=OPS[n], name=n, description=d)
            for n, d in descriptions.items()]


POOR = {
    "lookup_ticket": "gets ticket data",
    "runbook_for":   "gets runbook data",
    "find_similar":  "finds things",
}

def good_descriptions() -> dict:
    """Same three functions. Tell the model what each returns and when to prefer it."""
    return {
        "lookup_ticket": ("Return one support ticket by its reference, e.g. 'TCK-4001': customer, "
                          "product, version, severity and error code. Start here when you are "
                          "given a ticket reference and do not yet know what is wrong."),
        "runbook_for":   ("Return what support is ALLOWED to do about one error code, e.g. "
                          "'VPN-513'. Call this once you know the error code and need the action, "
                          "including whether it must be escalated."),
        "find_similar":  ("Return the references of other tickets reporting the same error code. "
                          "Use this to judge how widespread an issue is -- not to find out what "
                          "to do about it, which is runbook_for."),
    }

In [ ]:
# --- Self-check: Section 1   (real StructuredTool objects -- built, not invoked, so no model)
def tool_schema(descriptions):
    return {t.name: t.description for t in build_tools(descriptions)}

check("three real tools are built from the words you wrote",
      lambda: set(tool_schema(good_descriptions())) == set(OPS))
check("each description says what the tool returns, not just that it exists",
      lambda: all(len(d) > 60 for d in tool_schema(good_descriptions()).values()),
      "'gets ticket data' is the version we are measuring against -- say what comes back")
check("find_similar distinguishes itself from runbook_for",
      lambda: "runbook" in tool_schema(good_descriptions())["find_similar"].lower(),
      "two tools taking the same argument is exactly where a model guesses; say which is which")
score()

## Section 2 &mdash; A typed answer, not a paragraph

A paragraph has to be parsed by whatever comes next. A schema does not. Declare what a resolution
*is*, and the model fills it in.

The escalation rule is the interesting field: it must come from the runbook, not from the model's
judgement about how serious the ticket sounds.

In [ ]:
from pydantic import BaseModel, Field

class Resolution(BaseModel):
    """What support decided about one ticket."""
    ticket: str    = Field(description="The ticket reference, e.g. 'TCK-4001'")
    action: str    = Field(description="The single next action, in one imperative sentence")
    escalate: bool = Field(description="True only if the runbook says this must go to another "
                                       "desk, not if the ticket merely sounds serious")


def expected_escalation(ref: str) -> bool:
    """Ground truth for the eval: which tickets must NOT be resolved by support."""
    return TICKETS[ref]["error_code"] in MUST_ESCALATE

In [ ]:
# --- Self-check: Section 2   (the schema, and the ground truth -- no model yet)
def described(field):
    d = Resolution.model_fields[field].description
    if d.strip() == "BLANK":
        raise NameError(f"Resolution.{field} description is still BLANK")   # -> [TODO]
    return d

check("action says what to write, not just that something goes there",
      lambda: len(described("action")) > 25)
check("escalate is tied to the runbook rather than to how the ticket sounds",
      lambda: "runbook" in described("escalate").lower(),
      "'high severity' and 'must escalate' are different things -- TCK-4001 is high and resolvable")
check("only the security ticket must be escalated",
      lambda: [r for r in sorted(TICKETS) if expected_escalation(r)] == ["TCK-4004"])
score()

## Run it for real &mdash; the measurement

Five questions, each with one obviously correct first tool. Same model, same functions, twice.

In [ ]:
EVAL = [
    ("What is wrong with TCK-4001?",                       "lookup_ticket"),
    ("What are we allowed to do about error VPN-513?",     "runbook_for"),
    ("How many customers are hitting APP-002?",            "find_similar"),
    ("Who raised TCK-4004?",                               "lookup_ticket"),
    ("Is SEC-900 something support can close?",            "runbook_for"),
]

def first_tool(question, tools):
    """Which tool does the model reach for first? Given -- you are measuring, not building."""
    msg = get_llm().bind_tools(tools).invoke(
        [("system", "You are a tech support analyst. Use a tool."), ("human", question)])
    return msg.tool_calls[0]["name"] if msg.tool_calls else "(none)"

def score_arm(descriptions):
    tools = build_tools(descriptions)
    hits = [(q, want, first_tool(q, tools)) for q, want in EVAL]
    return sum(w == g for _, w, g in hits), hits


if llm_ready():
    def measure():
        for label, desc in [("poor", POOR), ("good", good_descriptions())]:
            n, hits = score_arm(desc)
            print(f"--- {label} descriptions: {n}/{len(EVAL)} first tools correct")
            for q, want, got in hits:
                print(f"    {'ok ' if want == got else 'MISS'} {q[:46]:48} want={want:14} got={got}")
    guard(measure)

### Read it

Same model. Same three Python functions. Only the words changed.

**What the poor arm gets wrong is not random.** It confuses `runbook_for` and `find_similar` &mdash;
two tools taking the same argument, described in a way that does not distinguish them. That is
where a model always guesses, and it is the thing your descriptions have to fix.

**This is the cheapest accuracy in the course.** No model change, no extra call, no new framework
&mdash; and it is the first thing to check when an agent picks the wrong tool. Module 4 does this
properly against a larger eval set; the habit starts here.

In [ ]:
score()

## Your turn

1. Make `find_similar`'s description say only &ldquo;searches tickets&rdquo; and re-run. Which
   questions move? Bad descriptions fail in a *predictable* place, which is why this is debuggable.
2. Add a fourth tool that genuinely overlaps &mdash; `ticket_history(ref)` &mdash; and get the
   agent to five out of five again. The fix is in the words, not the code.